# 07 - Create AI Agent via Project REST

Create an Azure AI Foundry project-scoped agent using REST API calls.

Pre-requisite: run 01_deploy_infra.ipynb and 02_configure.ipynb first.

---

## Step 1 - Load environment and auth

In [ ]:
import json
import os
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

project_endpoint = os.environ["AZURE_FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")
project_name = os.environ.get("AZURE_FOUNDRY_PROJECT_NAME", "")
deployment = os.environ["AZURE_AI_DEPLOYMENT"]
auth_mode = os.environ.get("AZURE_AUTH_MODE", "aad").lower()
if auth_mode != "aad":
    raise RuntimeError(f"Unsupported auth mode for this demo: {auth_mode}. Expected 'aad'.")

credential = AzureCliCredential()

# services.ai.azure.com often uses ai.azure.com scope; keep cognitive scope as fallback.
def get_project_token():
    try:
        return credential.get_token("https://ai.azure.com/.default").token
    except Exception:
        return credential.get_token("https://cognitiveservices.azure.com/.default").token

_ = get_project_token()

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

print(f"Project endpoint : {project_endpoint}")
print(f"Project name     : {project_name}")
print(f"Deployment       : {deployment}")
print("Credential       : AzureCliCredential")

Project endpoint : https://aigpt4o01tgrzn.services.ai.azure.com/api/projects/proj-gpt4o01
Project name     : proj-gpt4o01
Deployment       : gpt-4o
Credential       : AzureCliCredential


## Step 3 - Create an agent

In [15]:
agent_name = "quick-rest-agent"
agent_instructions = "You are a helpful AI agent for quick prototyping."

create_url = f"{project_endpoint}/agents/{agent_name}/versions"
create_payload = {
    "definition": {
        "kind": "prompt",
        "model": deployment,
        "instructions": agent_instructions,
        "tools": [{"type": "code_interpreter"}],
    }
}

create_response = requests.post(
    create_url,
    params={"api-version": "v1"},
    headers={
        "Authorization": f"Bearer {get_project_token()}",
        "Content-Type": "application/json",
    },
    json=create_payload,
    timeout=60,
 )

if create_response.status_code >= 400:
    raise RuntimeError(
        f"Project agent creation failed: HTTP {create_response.status_code} (v1) - {create_response.text}"
    )

created_payload = create_response.json()
created_agent_name = (
    created_payload.get("agent_name")
    or created_payload.get("name")
    or agent_name
)

print(f"Created agent : {created_agent_name}")
print(f"Create URL    : {create_url}")
print("API version   : v1")
print(json.dumps(created_payload, indent=2))

Created agent : quick-rest-agent
Create URL    : https://aigpt4o01tgrzn.services.ai.azure.com/api/projects/proj-gpt4o01/agents/quick-rest-agent/versions
API version   : v1
{
  "metadata": {},
  "object": "agent.version",
  "id": "quick-rest-agent:1",
  "name": "quick-rest-agent",
  "version": "1",
  "description": "",
  "created_at": 1782147202,
  "definition": {
    "kind": "prompt",
    "model": "gpt-4o",
    "instructions": "You are a helpful AI agent for quick prototyping.",
    "tools": [
      {
        "type": "code_interpreter"
      }
    ]
  },
  "status": "active",
  "instance_identity": {
    "principal_id": "15af9a2a-7c7b-4a89-8017-d5802f2f3f77",
    "client_id": "15af9a2a-7c7b-4a89-8017-d5802f2f3f77"
  },
  "blueprint": {
    "principal_id": "d1d62216-d57b-4859-94d6-207a472a3e06",
    "client_id": "f6fc8d8e-8ad9-493a-97ed-2daf410029dd"
  },
  "blueprint_reference": {
    "type": "ManagedAgentIdentityBlueprint",
    "blueprint_id": "quick-rest-agent-f4989"
  },
  "agent_gu

## Step 4 - Call the created agent

Invoke the project agent through its OpenAI protocol endpoint using the Responses API.

In [16]:
created_agent_name = (
    created_payload.get("agent_name")
    or created_payload.get("name")
    or agent_name
)

invoke_url = f"{project_endpoint}/agents/{created_agent_name}/endpoint/protocols/openai/responses"
invoke_payload = {
    "input": "Give me a 2-sentence summary of what this demo notebook does.",
    "max_output_tokens": 256,
}

invoke_response = requests.post(
    invoke_url,
    params={"api-version": "v1"},
    headers={
        "Authorization": f"Bearer {get_project_token()}",
        "Content-Type": "application/json",
    },
    json=invoke_payload,
    timeout=90,
 )

if invoke_response.status_code >= 400:
    raise RuntimeError(
        f"Agent invoke failed: HTTP {invoke_response.status_code} (v1) - {invoke_response.text}"
    )

invocation_payload = invoke_response.json()
invocation_text = invocation_payload.get("output_text")

if not invocation_text:
    output_items = invocation_payload.get("output") or []
    if output_items:
        content_items = output_items[0].get("content") or []
        if content_items:
            invocation_text = content_items[0].get("text")

print(f"Invoked agent: {created_agent_name}")
print(f"Invoke URL   : {invoke_url}")
print("API version  : v1")
print("Agent reply:")
print(invocation_text or "(No text output found in response payload.)")

Invoked agent: quick-rest-agent
Invoke URL   : https://aigpt4o01tgrzn.services.ai.azure.com/api/projects/proj-gpt4o01/agents/quick-rest-agent/endpoint/protocols/openai/responses
API version  : v1
Agent reply:
This demo notebook allows for rapid prototyping by executing Python code in a stateful Jupyter notebook environment. It supports tasks like data analysis, visualization, file generation (e.g., presentations, reports, or datasets), and interaction with AI models for automated computations and content creation.


## Step 5 - Save response

In [17]:
output_file = output_dir / "07_create_agent_rest_result.json"
result = {
    "created_agent": created_payload,
    "invocation": invocation_payload,
}
output_file.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"Saved: {output_file.resolve()}")

Saved: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\outputs\07_create_agent_rest_result.json


---

## Agent creation complete

You can now reuse the returned agent id in follow-up REST operations.